# Single-threshold directional change

This first notebook builds the event ledger for one directional-change threshold. It deliberately stops before plotting, EDA, or regime modelling.

For each completed extreme-to-extreme trend:

$$\mathrm{TMV} = \frac{|P_{\mathrm{EXT},n} - P_{\mathrm{EXT},n-1}|}{P_{\mathrm{EXT},n-1}\,\theta}$$

$$T = t_{\mathrm{EXT},n} - t_{\mathrm{EXT},n-1}$$

$$R = \frac{\mathrm{TMV}\,\theta}{T}$$

The ending extreme is identified retrospectively. Therefore, TMV, T, and R are not available at `ext_end_idx`; they first become knowable at the later DC confirmation, `dcc_idx`. The notebook records that timestamp as `information_time`.

In [48]:
from pathlib import Path

import numpy as np
import pandas as pd


from dc_basics.core import parse_dc_events
from dc_basics.plotting import plot_dc_indicator_distributions, plot_directional_change

## Configuration

In [49]:
DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "US500.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"

THETA = 0.003
N_PRICES = 5_000

print(f"Symbol: {SYMBOL}")
print(f"Data: {DATA_PATH}")
print(f"Threshold: {THETA:.4%}")
print(f"Chart window: {N_PRICES:,} prices")

Symbol: US500.r
Data: D:\data_dump\market_data\raw\FXPIG-Server\5\US500.r.parquet
Threshold: 0.3000%
Chart window: 5,000 prices


## Load and audit the bars

Directional-change parsing is path-dependent, so the price series must be chronological, unique, finite, and strictly positive. If `time` is stored as a column rather than the Parquet index, it is promoted to the index explicitly.

In [50]:
raw = pd.read_parquet(DATA_PATH)

raw = raw.sort_index()
prices = raw["close"].astype(np.float64)

print(f"Bars: {len(prices):,}")
print(f"Start: {prices.index[0]}")
print(f"End: {prices.index[-1]}")
print(f"Timezone: {prices.index.tz}")
raw.tail()

Bars: 228,362
Start: 2023-05-22 14:00:00+02:00
End: 2026-08-13 13:30:00+02:00
Timezone: pytz.FixedOffset(120)


,open,high,low,close,tick_volume,spread
date,,,,,,
2026-08-13 13:10:00+02:00,7765.89,7766.64,7764.52,7766.40,185.0,41
2026-08-13 13:15:00+02:00,7766.52,7766.89,7763.90,7765.77,241.0,41
2026-08-13 13:20:00+02:00,7765.77,7768.27,7764.77,7767.77,183.0,41
2026-08-13 13:25:00+02:00,7767.77,7769.02,7767.77,7768.40,153.0,41
2026-08-13 13:30:00+02:00,7768.52,7768.77,7767.27,7768.39,216.0,41


## Calculate completed directional-change events

`parse_dc_events` returns one row per completed trend. `event_id` gives each event a stable notebook identifier. `information_time` is deliberately copied from `dcc_idx`, because that confirmation bar is when the ending extreme and its completed TMV, T, and R become knowable.

In [51]:
events = parse_dc_events(prices=prices, theta=THETA).copy()

if events.empty:
    raise ValueError("No completed DC events were found. Check the data and threshold.")

events.insert(0, "event_id", np.arange(1, len(events) + 1, dtype=np.int64))
events.insert(1, "threshold", THETA)
events.insert(2, "information_time", events["dcc_idx"])

event_columns = [
    "event_id", "threshold", "direction", "ext_start_idx", "ext_start_price",
    "ext_end_idx", "ext_end_price", "dcc_idx", "dcc_price",
    "information_time", "tmv", "t", "r",
]

print(f"Completed events: {len(events):,}")
events[event_columns].tail(10)

Completed events: 3,649


,event_id,threshold,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,dcc_idx,dcc_price,information_time,tmv,t,r
3639,3640,0.003,down,2026-08-05 18:55:00+02:00,7800.92,2026-08-06 22:25:00+02:00,7710.47,2026-08-07 16:05:00+02:00,7734.12,2026-08-07 16:05:00+02:00,3.864929,318,0.000036
3640,3641,0.003,up,2026-08-06 22:25:00+02:00,7710.47,2026-08-07 17:50:00+02:00,7760.25,2026-08-07 19:00:00+02:00,7734.87,2026-08-07 19:00:00+02:00,2.152052,221,0.000029
3641,3642,0.003,down,2026-08-07 17:50:00+02:00,7760.25,2026-08-07 19:10:00+02:00,7728.37,2026-08-07 19:20:00+02:00,7751.62,2026-08-07 19:20:00+02:00,1.369372,16,0.000257
3642,3643,0.003,up,2026-08-07 19:10:00+02:00,7728.37,2026-08-07 20:25:00+02:00,7768.12,2026-08-07 23:30:00+02:00,7744.37,2026-08-07 23:30:00+02:00,1.714462,15,0.000343
3643,3644,0.003,down,2026-08-07 20:25:00+02:00,7768.12,2026-08-07 23:40:00+02:00,7739.87,2026-08-10 06:15:00+02:00,7763.24,2026-08-10 06:15:00+02:00,1.212220,39,0.000093
3644,3645,0.003,up,2026-08-07 23:40:00+02:00,7739.87,2026-08-10 19:45:00+02:00,7780.87,2026-08-10 21:35:00+02:00,7757.37,2026-08-10 21:35:00+02:00,1.765749,229,0.000023
3645,3646,0.003,down,2026-08-10 19:45:00+02:00,7780.87,2026-08-10 21:40:00+02:00,7752.12,2026-08-11 16:00:00+02:00,7779.72,2026-08-11 16:00:00+02:00,1.231653,23,0.000161
3646,3647,0.003,up,2026-08-10 21:40:00+02:00,7752.12,2026-08-11 16:00:00+02:00,7779.72,2026-08-11 20:15:00+02:00,7755.72,2026-08-11 20:15:00+02:00,1.186772,208,0.000017
3647,3648,0.003,down,2026-08-11 16:00:00+02:00,7779.72,2026-08-11 23:25:00+02:00,7726.72,2026-08-12 14:00:00+02:00,7750.07,2026-08-12 14:00:00+02:00,2.270862,89,0.000077
3648,3649,0.003,up,2026-08-11 23:25:00+02:00,7726.72,2026-08-12 18:25:00+02:00,7777.07,2026-08-12 19:00:00+02:00,7750.07,2026-08-12 19:00:00+02:00,2.172116,216,0.000030


## Visualize the last N prices

The close is shown with completed extreme-to-extreme trends and their later DC confirmations. A turning-point marker describes where an extreme occurred; its hover text shows when that extreme became knowable. The final unfinished event is intentionally absent because `parse_dc_events` emits completed events only.

In [52]:
fig = plot_directional_change(prices=prices, events=events, n_prices=N_PRICES, title=f"{SYMBOL} | theta={THETA:.4%}",)
fig.show()

## Descriptive EDA of TMV, T, and R

The mean answers the average question, but these event indicators are typically right-skewed. Median and tail quantiles are reported beside it so a small number of extreme events cannot silently determine the interpretation. These are descriptive event-level results: consecutive DC events are path-dependent, so the rows must not be treated as independent observations for significance testing.

In [53]:
INDICATORS = ["tmv", "t", "r"]
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

overall_summary = events[INDICATORS].describe(percentiles=QUANTILES).T
overall_summary.index.name = "indicator"
overall_summary.style.format(precision=6)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
indicator,,,,,,,,,,,,
tmv,3649.000000,2.408017,1.557137,1.000390,1.132111,1.380040,1.922709,2.908315,4.224411,5.240841,8.070557,21.481858
t,3649.000000,62.521787,90.223514,1.000000,5.000000,11.000000,27.000000,74.000000,178.000000,239.600000,410.600000,1100.000000
r,3649.000000,0.000522,0.000936,0.000009,0.000045,0.000088,0.000223,0.000516,0.001257,0.002052,0.004731,0.011954


### Direction-conditioned averages

The event count must be read with every directional comparison. A difference between up and down averages is descriptive here; it is not yet evidence of a stable or tradable asymmetry.

In [54]:
direction_summary = events.groupby("direction", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_mean=("tmv", "mean"),
    tmv_median=("tmv", "median"),
    t_mean=("t", "mean"),
    t_median=("t", "median"),
    r_mean=("r", "mean"),
    r_median=("r", "median"),
)
direction_summary.style.format({
    "event_count": "{:,}",
    "tmv_mean": "{:.6f}",
    "tmv_median": "{:.6f}",
    "t_mean": "{:.2f}",
    "t_median": "{:.2f}",
    "r_mean": "{:.6e}",
    "r_median": "{:.6e}",
})

,event_count,tmv_mean,tmv_median,t_mean,t_median,r_mean,r_median
direction,,,,,,,
up,"1,825",2.476115,1.946773,72.94,29.00,5.145303e-04,2.051897e-04
down,"1,824",2.339882,1.904402,52.10,25.50,5.294216e-04,2.407503e-04


### Direction-conditioned distributions

Each histogram is normalized within direction, so up and down shapes remain comparable even if their event counts differ. The initial x-axis view ends at the full-sample 99th percentile; Plotly's **Autoscale** button reveals all tail observations.

In [55]:
distribution_fig = plot_dc_indicator_distributions(events)
distribution_fig.show()

## Find events whose extremes are consecutive observations

`T = 1` means the starting and ending extremes occupy adjacent rows in the stored close series. `elapsed_minutes` checks whether those observations are also one 5-minute period apart in wall-clock time. `confirmation_delay_bars` measures how many subsequent observations were needed to confirm the ending extreme.

In [56]:
consecutive_events = events.loc[events["t"] == 1].copy()
consecutive_events["elapsed_minutes"] = (consecutive_events["ext_end_idx"] - consecutive_events["ext_start_idx"]).dt.total_seconds() / 60.0
consecutive_events["observed_move_pct"] = (consecutive_events["ext_end_price"] / consecutive_events["ext_start_price"] - 1.0) * 100.0
consecutive_events["confirmation_delay_bars"] = (consecutive_events["dcc_pos"] - consecutive_events["ext_end_pos"])

consecutive_columns = [
    "event_id", "direction", "ext_start_idx", "ext_start_price",
    "ext_end_idx", "ext_end_price", "elapsed_minutes", "observed_move_pct",
    "tmv", "r", "dcc_idx", "confirmation_delay_bars",
]

print(f"Events with T=1: {len(consecutive_events):,} of {len(events):,}")
consecutive_events[consecutive_columns]

Events with T=1: 59 of 3,649


,event_id,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,elapsed_minutes,observed_move_pct,tmv,r,dcc_idx,confirmation_delay_bars
617,618,down,2024-02-02 19:35:00+02:00,4960.44,2024-02-02 19:40:00+02:00,4941.44,5.0,-0.383031,1.276768,0.003830,2024-02-02 20:15:00+02:00,7
691,692,down,2024-03-20 21:25:00+02:00,5202.49,2024-03-20 21:30:00+02:00,5186.14,5.0,-0.314273,1.047575,0.003143,2024-03-20 21:40:00+02:00,2
923,924,down,2024-07-19 00:45:00+02:00,5562.82,2024-07-19 00:50:00+02:00,5544.77,5.0,-0.324476,1.081586,0.003245,2024-07-19 03:05:00+02:00,15
992,993,up,2024-08-02 18:30:00+02:00,5358.32,2024-08-02 18:35:00+02:00,5381.32,5.0,0.429239,1.430797,0.004292,2024-08-02 18:55:00+02:00,4
1017,1018,down,2024-08-05 18:35:00+02:00,5174.47,2024-08-05 18:40:00+02:00,5158.92,5.0,-0.300514,1.001713,0.003005,2024-08-05 18:45:00+02:00,1
1018,1019,up,2024-08-05 18:40:00+02:00,5158.92,2024-08-05 18:45:00+02:00,5186.54,5.0,0.535383,1.784611,0.005354,2024-08-05 18:55:00+02:00,2
1020,1021,up,2024-08-05 18:55:00+02:00,5169.32,2024-08-05 19:00:00+02:00,5210.82,5.0,0.802814,2.676045,0.008028,2024-08-05 19:05:00+02:00,1
1027,1028,down,2024-08-06 01:05:00+02:00,5218.12,2024-08-06 01:10:00+02:00,5202.37,5.0,-0.301833,1.006109,0.003018,2024-08-06 01:55:00+02:00,9
1042,1043,up,2024-08-07 00:15:00+02:00,5275.22,2024-08-07 00:20:00+02:00,5292.97,5.0,0.336479,1.121596,0.003365,2024-08-07 00:40:00+02:00,4
1671,1672,down,2025-03-05 22:00:00+02:00,5824.27,2025-03-05 22:05:00+02:00,5799.02,5.0,-0.433531,1.445102,0.004335,2025-03-05 22:45:00+02:00,8


### Inspect one consecutive-extreme event on the raw bars

Change `EVENT_ID` to inspect another row from the table above. The displayed path begins at the starting extreme and ends at the later confirmation bar.

In [57]:
if consecutive_events.empty:
    raise ValueError("No T=1 events are available at the current threshold.")

EVENT_ID = int(consecutive_events.iloc[-1]["event_id"])
selected_event = events.loc[events["event_id"] == EVENT_ID].iloc[0]

event_bars = raw.iloc[
    int(selected_event["ext_start_pos"]):int(selected_event["dcc_pos"]) + 1
].copy()
event_bars["dc_landmark"] = ""
event_bars.loc[selected_event["ext_start_idx"], "dc_landmark"] = "start EXT"
event_bars.loc[selected_event["ext_end_idx"], "dc_landmark"] = "end EXT"
event_bars.loc[selected_event["dcc_idx"], "dc_landmark"] = "DCC confirmation"

print(
    f"Event {EVENT_ID}: {selected_event['direction']} | "
    f"T={selected_event['t']} | TMV={selected_event['tmv']:.4f} | R={selected_event['r']:.3e}"
)
event_bars[["open", "high", "low", "close", "dc_landmark"]]

Event 3453: up | T=1 | TMV=1.0046 | R=3.014e-03


,open,high,low,close,dc_landmark
date,,,,,
2026-06-11 19:15:00+02:00,7301.02,7307.27,7292.02,7293.39,start EXT
2026-06-11 19:20:00+02:00,7292.77,7319.62,7292.40,7315.37,end EXT
2026-06-11 19:25:00+02:00,7315.87,7320.62,7299.87,7310.87,
2026-06-11 19:30:00+02:00,7311.12,7316.62,7300.12,7304.12,
2026-06-11 19:35:00+02:00,7304.37,7323.62,7304.12,7314.62,
2026-06-11 19:40:00+02:00,7314.62,7321.74,7308.37,7309.87,
2026-06-11 19:45:00+02:00,7309.99,7313.12,7301.62,7303.37,
2026-06-11 19:50:00+02:00,7303.50,7305.37,7293.87,7294.62,
2026-06-11 19:55:00+02:00,7295.87,7296.37,7274.12,7276.62,DCC confirmation
